# PhaseCoder Demo: Dynamic Microphone Array Extension

End-to-end demonstration of:
1. Generating a synthetic spatial audio dataset with pyroomacoustics
2. Training a baseline PhaseCoder model (static-array assumption)
3. Training an IMU-conditioned PhaseCoder that handles array rotation
4. Comparing both models on rotation-rate-binned validation data

**Methodology:** Same as Dementyev et al. 2026 — RIR-based synthesis, magnitude+phase STFT,
geometry-agnostic positional embeddings, transformer encoder, three classification heads.
Only the *scale* is reduced for demo feasibility.

## How to use this notebook

- Run cells sequentially top to bottom.
- The dataset configuration block lets you scale from quick smoke tests (500 clips, ~3 min)
  up to serious training (100k clips, ~1 hour generation + 6+ hours training).
- All cells are GPU-aware; if no GPU is available, training will run on CPU (much slower).

## Required files in the same directory

- `PhaseCoder.py` — the model
- `imu_preprocessing.py` — IMU utility (loaded for reference, not directly called here)


## 1. Setup

Install dependencies and import everything we'll need.

In [2]:
# Install dependencies (skip if already installed)
# Uncomment the next line if running for the first time:
# !pip install torch numpy pyroomacoustics matplotlib

import os
import math
import json
import time
from pathlib import Path
from multiprocessing import Pool, cpu_count

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import pyroomacoustics as pra

# Import the PhaseCoder model and loss from your existing implementation
from PhaseCoder import PhaseCoder, PhaseCoderLoss

# Confirm GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Available memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected — training will be slow on CPU.")
    print("   Consider reducing num_clips and epochs for testing.")


Using device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
Available memory: 8.6 GB


## 2. Configuration

**This is the main knob you'll adjust.** Pick a preset based on how much time/compute you have:

| Preset | num_clips | Time on GPU | Time on CPU | Use for |
|--------|-----------|-------------|-------------|---------|
| Smoke test | 500 | ~5 min | ~30 min | Pipeline sanity check |
| Quick demo | 5,000 | ~30 min | ~3 hours | Initial results |
| Solid demo | 25,000 | ~2 hours | overnight | Presentation-ready |
| Full scale | 100,000 | ~6-9 hours | days | Serious results |

In [ ]:
# ============================================================
# CONFIGURATION — adjust based on your time/compute budget
# ============================================================

# Dataset size
NUM_CLIPS = 100,000           # Try 500 for smoke test, 25_000 for solid demo
FRACTION_DYNAMIC = 0.5      # Fraction of clips with rotating array
VAL_FRACTION = 0.05         # 5% held out for validation

# Output paths
DATA_DIR = Path("./demo_data")
BASELINE_RUN_DIR = Path("./runs/baseline")
IMU_RUN_DIR = Path("./runs/imu")
EVAL_DIR = Path("./eval_results")

# Training hyperparameters
EPOCHS = 30                 # 15-30 typically; more data → fewer epochs needed
BATCH_SIZE = 32 if device.type == "cuda" else 16
LEARNING_RATE = 5e-4
WARMUP_EPOCHS = 2
NUM_DATALOADER_WORKERS = 2

# Generator parallelism
NUM_GENERATION_WORKERS = max(1, cpu_count() - 1)

# Random seed for reproducibility
SEED = 0

print(f"Configuration:")
print(f"  NUM_CLIPS:        {NUM_CLIPS:,}")
print(f"  FRACTION_DYNAMIC: {FRACTION_DYNAMIC}")
print(f"  EPOCHS:           {EPOCHS}")
print(f"  BATCH_SIZE:       {BATCH_SIZE}")
print(f"  Generation workers: {NUM_GENERATION_WORKERS}")
print(f"  Output dirs:      {DATA_DIR}, {BASELINE_RUN_DIR}, {IMU_RUN_DIR}")


## 3. PhaseCoder constants and helper functions

These match the paper exactly: 16 kHz, 250 ms clips, STFT with 256 win / 128 hop,
38 azimuth bins / 18 elevation bins / 13 distance bins.

In [ ]:
# Constants matching PhaseCoder paper
SAMPLE_RATE = 16000
CLIP_DURATION_S = 0.25
NUM_SAMPLES = int(SAMPLE_RATE * CLIP_DURATION_S)  # 4000
N_FFT = 256
HOP_LENGTH = 128
NUM_FRAMES = 32  # torch.stft(center=True) on 4000 samples → 32 frames

NUM_AZIMUTH = 38
NUM_ELEVATION = 18
NUM_DISTANCE = 13

DISTANCE_BIN_EDGES = np.array([0.4, 0.6, 0.8, 1.0, 1.3, 1.6, 2.0, 2.5, 3.0, 3.5, 4.5, 5.5, 7.0])

print(f"Audio config: {SAMPLE_RATE}Hz, {CLIP_DURATION_S*1000:.0f}ms clips, {NUM_FRAMES} STFT frames")
print(f"Label space: {NUM_AZIMUTH} azimuth × {NUM_ELEVATION} elevation × {NUM_DISTANCE} distance bins")


In [ ]:
def generate_source_signal(duration_s: float, rng: np.random.Generator) -> np.ndarray:
    """Synthetic harmonic source with broadband content for STFT."""
    n = int(SAMPLE_RATE * duration_s)
    t = np.arange(n) / SAMPLE_RATE

    f0 = rng.uniform(120, 350)
    signal = np.zeros(n)
    for harmonic in range(1, 7):
        amp = rng.uniform(0.3, 1.0) / harmonic
        phase = rng.uniform(0, 2 * np.pi)
        signal += amp * np.sin(2 * np.pi * f0 * harmonic * t + phase)

    am = 0.5 + 0.5 * np.sin(2 * np.pi * rng.uniform(2, 6) * t)
    signal *= am

    fade = int(0.015 * SAMPLE_RATE)
    envelope = np.ones(n)
    envelope[:fade] = np.linspace(0, 1, fade)
    envelope[-fade:] = np.linspace(1, 0, fade)
    signal *= envelope

    signal /= max(np.max(np.abs(signal)), 1e-8)
    return (signal * 0.8).astype(np.float32)


def generate_random_array(rng: np.random.Generator) -> np.ndarray:
    """Random circular mic array with 4-8 mics, varying radius."""
    num_mics = rng.integers(4, 9)
    radius = rng.uniform(0.04, 0.10)
    angles = np.linspace(0, 2 * np.pi, num_mics, endpoint=False)
    angles += rng.uniform(-0.1, 0.1, size=num_mics)
    coords = np.stack([
        radius * np.cos(angles),
        radius * np.sin(angles),
        rng.uniform(-0.01, 0.01, size=num_mics),
    ], axis=-1)
    return coords.astype(np.float32)


def spherical_to_cartesian(az_deg, el_deg, distance):
    az = math.radians(az_deg); el = math.radians(el_deg)
    return np.array([
        distance * math.cos(el) * math.cos(az),
        distance * math.cos(el) * math.sin(az),
        distance * math.sin(el),
    ], dtype=np.float32)


def discretize_labels(az_deg, el_deg, dist):
    az_bin = min(int(az_deg / 360 * NUM_AZIMUTH), NUM_AZIMUTH - 1)
    el_bin = min(max(int((el_deg + 90) / 180 * NUM_ELEVATION), 0), NUM_ELEVATION - 1)
    dist_bin = min(int(np.searchsorted(DISTANCE_BIN_EDGES, dist)), NUM_DISTANCE - 1)
    return {"azimuth": az_bin, "elevation": el_bin, "distance": dist_bin}


def axis_angle_to_quaternion(axis, angle_rad):
    axis = axis / (np.linalg.norm(axis) + 1e-12)
    half = angle_rad / 2
    s = math.sin(half)
    return np.array([math.cos(half), axis[0]*s, axis[1]*s, axis[2]*s], dtype=np.float32)


def quaternion_to_rotation_matrix(q):
    w, x, y, z = q / (np.linalg.norm(q) + 1e-12)
    return np.array([
        [1-2*(y*y+z*z), 2*(x*y-z*w), 2*(x*z+y*w)],
        [2*(x*y+z*w), 1-2*(x*x+z*z), 2*(y*z-x*w)],
        [2*(x*z-y*w), 2*(y*z+x*w), 1-2*(x*x+y*y)],
    ], dtype=np.float32)

print("Helper functions defined.")


## 4. Clip simulation function

This simulates a single multichannel audio clip using pyroomacoustics' image-source method.
For dynamic clips, it generates per-frame quaternions reflecting the array's rotation trajectory.

In [ ]:
def simulate_clip(mic_coords, source_az_deg, source_el_deg, source_distance,
                  rng, rotation_rate_dps=0.0, rotation_axis=None):
    """Simulate one spatial audio clip with optional array rotation."""
    C = mic_coords.shape[0]

    # Random shoebox room
    room_dim = np.array([
        rng.uniform(4.0, 8.0), rng.uniform(3.0, 6.0), rng.uniform(2.5, 3.5),
    ])
    rt60 = rng.uniform(0.15, 0.35)
    e_absorption, max_order = pra.inverse_sabine(rt60, room_dim)
    room = pra.ShoeBox(room_dim, fs=SAMPLE_RATE,
                       materials=pra.Material(e_absorption), max_order=max_order)

    # Place array centroid at head height with margins
    centroid = np.array([
        rng.uniform(1.0, room_dim[0] - 1.0),
        rng.uniform(1.0, room_dim[1] - 1.0),
        rng.uniform(1.2, 1.8),
    ])

    # Compute IMU quaternions for dynamic clips
    if rotation_rate_dps > 0:
        if rotation_axis is None:
            rotation_axis = np.array([0, 0, 1], dtype=np.float32)
        rotation_axis = rotation_axis / np.linalg.norm(rotation_axis)
        frame_times = (np.arange(NUM_FRAMES) * HOP_LENGTH + N_FFT/2) / SAMPLE_RATE
        angles_rad = np.radians(rotation_rate_dps * frame_times)
        imu_quats = np.stack([axis_angle_to_quaternion(rotation_axis, a)
                               for a in angles_rad], axis=0)
        canonical_idx = NUM_FRAMES // 2
        R_canonical = quaternion_to_rotation_matrix(imu_quats[canonical_idx])
        mic_coords_rotated = mic_coords @ R_canonical.T
    else:
        imu_quats = None
        mic_coords_rotated = mic_coords

    mic_world = (centroid[None, :] + mic_coords_rotated).T
    room.add_microphone_array(pra.MicrophoneArray(mic_world, fs=SAMPLE_RATE))

    source_offset = spherical_to_cartesian(source_az_deg, source_el_deg, source_distance)
    source_world = np.clip(centroid + source_offset, 0.2, room_dim - 0.2)

    src_signal = generate_source_signal(CLIP_DURATION_S * 2.0, rng)
    room.add_source(source_world, signal=src_signal)
    room.simulate()

    sim_audio = room.mic_array.signals
    T_sim = sim_audio.shape[1]
    if T_sim >= NUM_SAMPLES:
        start = max(0, int(0.005 * SAMPLE_RATE))
        if start + NUM_SAMPLES > T_sim:
            start = T_sim - NUM_SAMPLES
        audio = sim_audio[:, start:start + NUM_SAMPLES]
    else:
        audio = np.zeros((C, NUM_SAMPLES), dtype=np.float32)
        audio[:, :T_sim] = sim_audio

    peak = np.max(np.abs(audio))
    if peak > 1e-8:
        audio = (audio / peak * 0.7).astype(np.float32)

    return audio.astype(np.float32), imu_quats

# Quick test of simulation
test_rng = np.random.default_rng(42)
test_mics = generate_random_array(test_rng)
test_audio, test_imu = simulate_clip(test_mics, 90.0, 0.0, 1.5, test_rng,
                                       rotation_rate_dps=200.0,
                                       rotation_axis=np.array([0, 0, 1.0]))
print(f"Test clip: audio shape {test_audio.shape}, IMU shape {test_imu.shape}")
print(f"Audio range: [{test_audio.min():.3f}, {test_audio.max():.3f}]")


## 5. Generate the dataset (parallel)

This is the slowest cell. Time depends on `NUM_CLIPS` and number of CPU cores.

**While this runs**, you can keep an eye on the progress output. It's safe to interrupt
(Kernel → Interrupt) if you want to test downstream cells with whatever clips have been
generated so far — the manifest is built from completed clips.

In [ ]:
def _generate_one_clip_worker(args):
    """Worker for multiprocessing pool."""
    clip_idx, output_dir_str, seed, fraction_dynamic = args
    output_dir = Path(output_dir_str)
    rng = np.random.default_rng(seed + clip_idx)

    try:
        mic_coords = generate_random_array(rng)
        source_az = float(rng.uniform(0, 360))
        source_el = float(rng.uniform(-30, 30))
        source_dist = float(np.exp(rng.uniform(np.log(0.5), np.log(5.0))))

        is_dynamic = rng.random() < fraction_dynamic
        if is_dynamic:
            rotation_rate = float(rng.choice([30, 60, 100, 150, 200, 300, 450])
                                  * rng.choice([-1, 1]))
            axis = np.array([
                rng.uniform(-0.3, 0.3), rng.uniform(-0.3, 0.3),
                rng.choice([-1, 1]) * rng.uniform(0.7, 1.0),
            ], dtype=np.float32)
            axis = axis / np.linalg.norm(axis)
        else:
            rotation_rate = 0.0
            axis = None

        audio, imu_quats = simulate_clip(
            mic_coords, source_az, source_el, source_dist, rng,
            rotation_rate_dps=abs(rotation_rate) if is_dynamic else 0.0,
            rotation_axis=axis)

        labels = discretize_labels(source_az, source_el, source_dist)
        clip_path = output_dir / f"clip_{clip_idx:06d}.npz"
        save_kwargs = {
            "audio": audio, "mic_coords": mic_coords,
            "azimuth_class": np.int64(labels["azimuth"]),
            "elevation_class": np.int64(labels["elevation"]),
            "distance_class": np.int64(labels["distance"]),
            "azimuth_deg": np.float32(source_az),
            "elevation_deg": np.float32(source_el),
            "distance_m": np.float32(source_dist),
            "is_dynamic": np.bool_(is_dynamic),
            "rotation_rate_dps": np.float32(rotation_rate),
        }
        if is_dynamic:
            save_kwargs["imu_quats"] = imu_quats
        np.savez_compressed(clip_path, **save_kwargs)

        return {
            "filename": clip_path.name, "is_dynamic": bool(is_dynamic),
            "rotation_rate_dps": float(rotation_rate),
            "azimuth_deg": float(source_az), "elevation_deg": float(source_el),
            "distance_m": float(source_dist), "num_mics": int(mic_coords.shape[0]),
        }
    except Exception:
        return None


In [ ]:
# Run dataset generation
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Generating {NUM_CLIPS:,} clips with {NUM_GENERATION_WORKERS} workers...")
print(f"Output: {DATA_DIR}")
print("-" * 60)

work_args = [(i, str(DATA_DIR), SEED, FRACTION_DYNAMIC) for i in range(NUM_CLIPS)]

start_time = time.time()
successful = []
failed = 0

# Note: in Jupyter, we use a Pool with the worker function defined at module scope.
# If running interactively and getting pickle errors, restart kernel and re-run.
with Pool(processes=NUM_GENERATION_WORKERS) as pool:
    for i, result in enumerate(pool.imap_unordered(_generate_one_clip_worker,
                                                     work_args, chunksize=16)):
        if result is not None:
            successful.append(result)
        else:
            failed += 1

        if (i + 1) % max(1, NUM_CLIPS // 20) == 0 or (i + 1) == NUM_CLIPS:
            elapsed = time.time() - start_time
            rate = (i + 1) / max(elapsed, 0.001)
            eta_min = (NUM_CLIPS - i - 1) / max(rate, 0.001) / 60
            print(f"  [{i+1:6d}/{NUM_CLIPS}] {100*(i+1)/NUM_CLIPS:5.1f}% | "
                  f"{rate:.1f} clips/sec | ETA: {eta_min:.1f} min")

total_min = (time.time() - start_time) / 60
print(f"\n✓ Generated {len(successful):,} clips ({failed} failed) in {total_min:.1f} min")

# Sort and split
successful.sort(key=lambda c: c["filename"])
rng = np.random.default_rng(SEED + 999_999)
n_val = int(len(successful) * VAL_FRACTION)
val_indices = set(rng.choice(len(successful), size=n_val, replace=False).tolist())

manifest = {
    "num_clips": len(successful),
    "fraction_dynamic": FRACTION_DYNAMIC,
    "sample_rate": SAMPLE_RATE,
    "num_frames": NUM_FRAMES,
    "num_azimuth": NUM_AZIMUTH,
    "num_elevation": NUM_ELEVATION,
    "num_distance": NUM_DISTANCE,
    "clips": successful,
    "train_filenames": [c["filename"] for i, c in enumerate(successful) if i not in val_indices],
    "val_filenames": [c["filename"] for i, c in enumerate(successful) if i in val_indices],
}

with open(DATA_DIR / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print(f"  Train: {len(manifest['train_filenames']):,}")
print(f"  Val:   {len(manifest['val_filenames']):,}")

# Statistics
dyn_count = sum(1 for c in successful if c["is_dynamic"])
print(f"\nStatistics:")
print(f"  Static:  {len(successful) - dyn_count:,}")
print(f"  Dynamic: {dyn_count:,}")


## 6. PyTorch Dataset and DataLoader

Custom collate function pads variable mic counts to the max in each batch.

In [ ]:
class ToyDataset(Dataset):
    def __init__(self, data_dir, filenames, use_imu):
        self.data_dir = data_dir
        self.filenames = filenames
        self.use_imu = use_imu

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        data = np.load(self.data_dir / self.filenames[idx])
        audio = torch.from_numpy(data["audio"]).float()
        mic_coords = torch.from_numpy(data["mic_coords"]).float()
        labels = {
            "azimuth": int(data["azimuth_class"]),
            "elevation": int(data["elevation_class"]),
            "distance": int(data["distance_class"]),
        }
        is_dynamic = bool(data["is_dynamic"])
        if self.use_imu and is_dynamic:
            imu_quats = torch.from_numpy(data["imu_quats"]).float()
        elif self.use_imu and not is_dynamic:
            F = data["imu_quats"].shape[0] if "imu_quats" in data.files else NUM_FRAMES
            imu_quats = torch.zeros(F, 4); imu_quats[:, 0] = 1.0
        else:
            imu_quats = None
        return {
            "audio": audio, "mic_coords": mic_coords, "imu_quats": imu_quats,
            "labels": labels, "is_dynamic": is_dynamic,
            "rotation_rate_dps": float(data["rotation_rate_dps"]),
        }


def collate_fn(batch):
    max_mics = max(item["mic_coords"].shape[0] for item in batch)
    B = len(batch)
    T = batch[0]["audio"].shape[-1]
    audio = torch.zeros(B, max_mics, T)
    mic_coords = torch.zeros(B, max_mics, 3)
    has_imu = batch[0]["imu_quats"] is not None
    imu_quats = torch.zeros(B, batch[0]["imu_quats"].shape[0], 4) if has_imu else None
    for i, item in enumerate(batch):
        c = item["audio"].shape[0]
        audio[i, :c] = item["audio"]
        mic_coords[i, :c] = item["mic_coords"]
        if c < max_mics:
            mic_coords[i, c:] = item["mic_coords"].mean(dim=0)
        if has_imu:
            imu_quats[i] = item["imu_quats"]
    labels = {k: torch.tensor([item["labels"][k] for item in batch], dtype=torch.long)
              for k in ["azimuth", "elevation", "distance"]}
    return {
        "audio": audio, "mic_coords": mic_coords, "imu_quats": imu_quats,
        "labels": labels,
        "is_dynamic": [item["is_dynamic"] for item in batch],
        "rotation_rate_dps": [item["rotation_rate_dps"] for item in batch],
    }

print("Dataset class defined.")


## 7. Training loop

Single function used to train both baseline and IMU models, parameterized by `use_imu`.
Includes warmup + cosine LR schedule, gradient clipping, and mixed-precision on GPU.

In [ ]:
def train_model(use_imu: bool, output_dir: Path, manifest, epochs=EPOCHS,
                batch_size=BATCH_SIZE, lr=LEARNING_RATE):
    """Train PhaseCoder with or without IMU conditioning."""
    output_dir.mkdir(parents=True, exist_ok=True)
    use_amp = device.type == "cuda"

    train_ds = ToyDataset(DATA_DIR, manifest["train_filenames"], use_imu=use_imu)
    val_ds = ToyDataset(DATA_DIR, manifest["val_filenames"], use_imu=use_imu)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              collate_fn=collate_fn, num_workers=NUM_DATALOADER_WORKERS,
                              drop_last=True, pin_memory=use_amp,
                              persistent_workers=NUM_DATALOADER_WORKERS > 0)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                            collate_fn=collate_fn, num_workers=max(1, NUM_DATALOADER_WORKERS // 2),
                            pin_memory=use_amp,
                            persistent_workers=NUM_DATALOADER_WORKERS > 0)

    model = PhaseCoder().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    def lr_lambda(epoch):
        if epoch < WARMUP_EPOCHS:
            return (epoch + 1) / WARMUP_EPOCHS
        progress = (epoch - WARMUP_EPOCHS) / max(1, epochs - WARMUP_EPOCHS)
        return 0.5 * (1 + np.cos(np.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    criterion = PhaseCoderLoss()
    scaler = torch.amp.GradScaler("cuda") if use_amp else None

    history = {"train_loss": [], "val_loss": [], "val_az_acc": [],
               "val_el_acc": [], "val_dist_acc": []}
    best_val = float("inf")
    overall_start = time.time()

    print(f"Training {'IMU' if use_imu else 'baseline'} model:")
    print(f"  Train: {len(train_ds):,} clips, Val: {len(val_ds):,} clips")
    print(f"  {sum(p.numel() for p in model.parameters())/1e6:.2f}M params, "
          f"batch_size={batch_size}, lr={lr}, epochs={epochs}")
    print("-" * 60)

    for epoch in range(epochs):
        model.train()
        train_losses = []
        epoch_start = time.time()

        for batch in train_loader:
            audio = batch["audio"].to(device, non_blocking=True)
            mic_coords = batch["mic_coords"].to(device, non_blocking=True)
            imu = batch["imu_quats"].to(device, non_blocking=True) if batch["imu_quats"] is not None else None
            targets = {k: v.to(device, non_blocking=True) for k, v in batch["labels"].items()}

            optimizer.zero_grad(set_to_none=True)
            if use_amp:
                with torch.amp.autocast("cuda", dtype=torch.float16):
                    out = model(audio, mic_coords, imu_orientations=imu)
                    losses = criterion(out, targets)
                scaler.scale(losses["loss"]).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer); scaler.update()
            else:
                out = model(audio, mic_coords, imu_orientations=imu)
                losses = criterion(out, targets)
                losses["loss"].backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            train_losses.append(losses["loss"].item())

        scheduler.step()

        # Validate
        model.eval()
        val_losses = []
        az_correct = el_correct = dist_correct = total = 0
        with torch.no_grad():
            for batch in val_loader:
                audio = batch["audio"].to(device, non_blocking=True)
                mic_coords = batch["mic_coords"].to(device, non_blocking=True)
                imu = batch["imu_quats"].to(device, non_blocking=True) if batch["imu_quats"] is not None else None
                targets = {k: v.to(device, non_blocking=True) for k, v in batch["labels"].items()}
                if use_amp:
                    with torch.amp.autocast("cuda", dtype=torch.float16):
                        out = model(audio, mic_coords, imu_orientations=imu)
                        losses = criterion(out, targets)
                else:
                    out = model(audio, mic_coords, imu_orientations=imu)
                    losses = criterion(out, targets)
                val_losses.append(losses["loss"].item())
                az_correct += (out["azimuth_logits"].argmax(dim=-1) == targets["azimuth"]).sum().item()
                el_correct += (out["elevation_logits"].argmax(dim=-1) == targets["elevation"]).sum().item()
                dist_correct += (out["distance_logits"].argmax(dim=-1) == targets["distance"]).sum().item()
                total += targets["azimuth"].shape[0]

        avg_train = float(np.mean(train_losses))
        avg_val = float(np.mean(val_losses))
        az_acc = az_correct / max(total, 1)
        el_acc = el_correct / max(total, 1)
        dist_acc = dist_correct / max(total, 1)

        history["train_loss"].append(avg_train)
        history["val_loss"].append(avg_val)
        history["val_az_acc"].append(az_acc)
        history["val_el_acc"].append(el_acc)
        history["val_dist_acc"].append(dist_acc)

        epoch_time = time.time() - epoch_start
        total_min = (time.time() - overall_start) / 60
        print(f"Epoch {epoch+1:3d}/{epochs} | train {avg_train:.3f} | val {avg_val:.3f} | "
              f"az {az_acc:.2%} el {el_acc:.2%} dist {dist_acc:.2%} | "
              f"{epoch_time:.0f}s ({total_min:.1f}min total)")

        if avg_val < best_val:
            best_val = avg_val
            torch.save({
                "model_state_dict": model.state_dict(),
                "epoch": epoch, "val_loss": avg_val,
                "val_az_acc": az_acc, "val_el_acc": el_acc, "val_dist_acc": dist_acc,
            }, output_dir / "best.pt")

    with open(output_dir / "history.json", "w") as f:
        json.dump(history, f, indent=2)

    print(f"\n✓ Done. Best val loss: {best_val:.3f}")
    return history

print("Training function defined.")


## 8. Train baseline model (no IMU)

This model ignores any IMU information — it sees dynamic clips as if they were static.
This is the "before" in our before/after comparison.

In [ ]:
# Reload manifest in case kernel was restarted
with open(DATA_DIR / "manifest.json") as f:
    manifest = json.load(f)

baseline_history = train_model(use_imu=False, output_dir=BASELINE_RUN_DIR, manifest=manifest)


## 9. Train IMU-conditioned model

Same architecture, same data, but this model receives per-frame quaternions for dynamic clips.
This is the "after" — the contribution of the dynamic-array extension.

In [ ]:
imu_history = train_model(use_imu=True, output_dir=IMU_RUN_DIR, manifest=manifest)


## 10. Training curves

Quick visual check that both models converged.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

epochs_x = np.arange(1, len(baseline_history["train_loss"]) + 1)

axes[0].plot(epochs_x, baseline_history["train_loss"], "--", label="Baseline train", color="#d62728", alpha=0.7)
axes[0].plot(epochs_x, baseline_history["val_loss"], "-", label="Baseline val", color="#d62728")
axes[0].plot(epochs_x, imu_history["train_loss"], "--", label="IMU train", color="#2ca02c", alpha=0.7)
axes[0].plot(epochs_x, imu_history["val_loss"], "-", label="IMU val", color="#2ca02c")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title("Training curves"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_x, baseline_history["val_az_acc"], "-o", label="Baseline azimuth", color="#d62728")
axes[1].plot(epochs_x, imu_history["val_az_acc"], "-o", label="IMU azimuth", color="#2ca02c")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Validation accuracy")
axes[1].set_title("Azimuth accuracy")
axes[1].axhline(y=1/NUM_AZIMUTH, color="gray", linestyle=":", label=f"Random ({100/NUM_AZIMUTH:.1f}%)")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()
print(f"Final azimuth accuracy: baseline={baseline_history['val_az_acc'][-1]:.2%}, "
      f"IMU={imu_history['val_az_acc'][-1]:.2%}")


## 11. Evaluation: baseline vs IMU by rotation rate

The headline figure of the demo. We bin validation clips by array rotation rate and compare
how each model handles each bucket. The expected pattern: baseline degrades as rotation increases,
IMU model holds up.

In [ ]:
def load_model_from_ckpt(ckpt_path):
    model = PhaseCoder().to(device)
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    return model


def evaluate(model, loader, use_imu):
    records = []
    with torch.no_grad():
        for batch in loader:
            audio = batch["audio"].to(device)
            mic_coords = batch["mic_coords"].to(device)
            imu = batch["imu_quats"].to(device) if (use_imu and batch["imu_quats"] is not None) else None
            targets = batch["labels"]
            out = model(audio, mic_coords, imu_orientations=imu)
            az_pred = out["azimuth_logits"].argmax(dim=-1).cpu()
            el_pred = out["elevation_logits"].argmax(dim=-1).cpu()
            dist_pred = out["distance_logits"].argmax(dim=-1).cpu()
            for i in range(audio.shape[0]):
                records.append({
                    "is_dynamic": batch["is_dynamic"][i],
                    "rotation_rate": abs(batch["rotation_rate_dps"][i]),
                    "az_correct": int(az_pred[i] == targets["azimuth"][i]),
                    "el_correct": int(el_pred[i] == targets["elevation"][i]),
                    "dist_correct": int(dist_pred[i] == targets["distance"][i]),
                })
    return records


def bin_by_rotation(records, bins=(0, 50, 150, 250, 400, 1000)):
    buckets = {}
    for r in records:
        for i in range(len(bins) - 1):
            if bins[i] <= r["rotation_rate"] < bins[i + 1]:
                key = f"{bins[i]}-{bins[i+1]}"
                buckets.setdefault(key, []).append(r)
                break
    return buckets


def acc(records):
    if not records: return {"az": 0, "el": 0, "dist": 0, "n": 0}
    n = len(records)
    return {"az": sum(r["az_correct"] for r in records)/n,
            "el": sum(r["el_correct"] for r in records)/n,
            "dist": sum(r["dist_correct"] for r in records)/n, "n": n}


# Run evaluation
baseline_model = load_model_from_ckpt(BASELINE_RUN_DIR / "best.pt")
imu_model = load_model_from_ckpt(IMU_RUN_DIR / "best.pt")

val_ds_static = ToyDataset(DATA_DIR, manifest["val_filenames"], use_imu=False)
val_ds_imu = ToyDataset(DATA_DIR, manifest["val_filenames"], use_imu=True)
val_loader_static = DataLoader(val_ds_static, batch_size=BATCH_SIZE, collate_fn=collate_fn, num_workers=0)
val_loader_imu = DataLoader(val_ds_imu, batch_size=BATCH_SIZE, collate_fn=collate_fn, num_workers=0)

print("Evaluating baseline...")
baseline_records = evaluate(baseline_model, val_loader_static, use_imu=False)
print("Evaluating IMU model...")
imu_records = evaluate(imu_model, val_loader_imu, use_imu=True)

baseline_buckets = bin_by_rotation(baseline_records)
imu_buckets = bin_by_rotation(imu_records)

# Print comparison table
print("\n" + "=" * 70)
print("RESULTS BY ROTATION RATE")
print("=" * 70)
print(f"{'Bucket (deg/sec)':<20} {'n':<6} {'Baseline Az':<14} {'IMU Az':<14} {'Δ':<10}")
print("-" * 70)
for k in sorted(baseline_buckets.keys(), key=lambda s: int(s.split("-")[0])):
    b = acc(baseline_buckets[k]); m = acc(imu_buckets.get(k, []))
    print(f"{k:<20} {b['n']:<6} {b['az']:<14.2%} {m['az']:<14.2%} {(m['az']-b['az']):+.2%}")

print(f"\nOverall:")
b_all = acc(baseline_records); m_all = acc(imu_records)
print(f"  Baseline:  az={b_all['az']:.2%}  el={b_all['el']:.2%}  dist={b_all['dist']:.2%}")
print(f"  IMU model: az={m_all['az']:.2%}  el={m_all['el']:.2%}  dist={m_all['dist']:.2%}")


## 12. The headline figure

The bar chart you'll show in the presentation. Save it to disk as a PNG.

In [ ]:
EVAL_DIR.mkdir(parents=True, exist_ok=True)

bucket_keys = sorted(set(baseline_buckets.keys()) | set(imu_buckets.keys()),
                     key=lambda s: int(s.split("-")[0]))
baseline_az = [acc(baseline_buckets.get(k, []))["az"] for k in bucket_keys]
imu_az = [acc(imu_buckets.get(k, []))["az"] for k in bucket_keys]
counts = [acc(baseline_buckets.get(k, []))["n"] for k in bucket_keys]

fig, ax = plt.subplots(figsize=(11, 6))
x = np.arange(len(bucket_keys)); width = 0.36

ax.bar(x - width/2, baseline_az, width, label="Baseline (no IMU)", color="#d62728", alpha=0.85)
ax.bar(x + width/2, imu_az, width, label="IMU-conditioned", color="#2ca02c", alpha=0.85)

ax.axhline(y=1/NUM_AZIMUTH, color="gray", linestyle=":",
           label=f"Random chance ({100/NUM_AZIMUTH:.1f}%)")

ax.set_xlabel("Rotation rate (deg/sec)", fontsize=12)
ax.set_ylabel("Azimuth classification accuracy", fontsize=12)
ax.set_title(f"PhaseCoder Performance vs. Array Rotation Rate\n"
             f"(Toy synthetic dataset, {len(manifest['train_filenames']):,} train clips)",
             fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels([f"{k}\n(n={n})" for k, n in zip(bucket_keys, counts)])
ax.legend(fontsize=11, loc="upper right")
ax.set_ylim(0, max(max(baseline_az + imu_az) * 1.15, 0.15))
ax.grid(axis="y", alpha=0.3)

for i, (b, m) in enumerate(zip(baseline_az, imu_az)):
    ax.text(i - width/2, b + 0.005, f"{b:.0%}", ha="center", fontsize=9)
    ax.text(i + width/2, m + 0.005, f"{m:.0%}", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig(EVAL_DIR / "comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {EVAL_DIR / 'comparison.png'}")


## 13. Save results

Write everything to disk in case you want to come back to it.

In [ ]:
results = {
    "config": {
        "num_clips": len(manifest["clips"]),
        "fraction_dynamic": FRACTION_DYNAMIC,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "lr": LEARNING_RATE,
    },
    "baseline_history": baseline_history,
    "imu_history": imu_history,
    "baseline_by_bucket": {k: acc(v) for k, v in baseline_buckets.items()},
    "imu_by_bucket": {k: acc(v) for k, v in imu_buckets.items()},
    "baseline_overall": acc(baseline_records),
    "imu_overall": acc(imu_records),
}

with open(EVAL_DIR / "results.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"Results saved to {EVAL_DIR / 'results.json'}")
print(f"\nSummary:")
print(f"  Dataset:   {len(manifest['clips']):,} clips ({len(manifest['train_filenames']):,} train, "
      f"{len(manifest['val_filenames']):,} val)")
print(f"  Baseline:  {results['baseline_overall']['az']:.2%} azimuth accuracy")
print(f"  IMU model: {results['imu_overall']['az']:.2%} azimuth accuracy")
print(f"  Δ:         {(results['imu_overall']['az'] - results['baseline_overall']['az']):+.2%}")


## What to highlight in the presentation

**Honest framing:** "This is a proof-of-concept on synthetic toy data, not a benchmark result.
It demonstrates that the architecture works end-to-end and that the IMU-conditioning extension
functions correctly."

**Key talking points:**
1. The full PhaseCoder methodology is preserved — RIR-based synthesis, geometry-agnostic embeddings,
   magnitude+phase STFT features, transformer encoder, three classification heads.
2. Only the *scale* is reduced (thousands of clips vs. paper's 4M).
3. The IMU-conditioning path trains and converges, demonstrating the design works.
4. Path forward: scale to LOCATA + AEA + custom synthetic data with full diversity.

**Caveats to acknowledge upfront:**
- Synthetic harmonic signals instead of real speech
- Limited room and geometry diversity vs. paper
- Small dataset means absolute accuracy is much lower than reported in paper
- This is the *first* result in a research program, not the *final* one

## Next steps

Once this demo is done, the priority list is:
1. Replace synthetic source signals with LibriSpeech subset
2. Scale dataset to 100k+ clips with broader room diversity
3. Run inference on LOCATA's moving-array tasks (Tasks 4-6) for real-world validation
4. Build the AEA-derived ground-truth pipeline for wearable evaluation
